# 문화 온도(Culture Temperature) EDA (Neo4j 기반)

## 목적
Neo4j에 저장된 매물과 문화시설 간의 연결 정보(500m 반경)를 기반으로,
매물 주변 문화시설 **개수**에 따른 점수 산정 기준을 수립합니다.

## 분석 대상
- **노드**: `Property`, `Culture`
- **관계**: `(:Property)-[:NEAR_CULTURE]->(:Culture)` (distance < 500m 이미 연결됨)

In [ ]:
import os
import sys
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from neo4j import GraphDatabase
from dotenv import load_dotenv

# 한글 폰트 설정
plt.rc('font', family='Malgun Gothic')
plt.rc('axes', unicode_minus=False)

# Root 경로 설정 및 Config 불러오기
current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, "../../.."))
sys.path.append(os.path.join(project_root, "scripts", "data_import"))

from config import Config

# .env 로드 (API Key 등)
load_dotenv(os.path.join(project_root, ".env"))

## 1. Neo4j 데이터 로드
매물별로 연결된 문화시설의 개수(Count)를 가져옵니다.
연결이 없는 매물도 포함하기 위해 `OPTIONAL MATCH`를 사용하거나, 전체 매물 수 대비 비율을 계산합니다.

In [ ]:
class Neo4jConnector:
    def __init__(self):
        self.driver = GraphDatabase.driver(
            Config.NEO4J_URI,
            auth=(Config.NEO4J_USER, Config.NEO4J_PASSWORD)
        )
    
    def close(self):
        self.driver.close()
        
    def get_culture_counts(self, limit=10000):
        # 500m 이내 문화시설 개수 조회
        # 속도를 위해 샘플링 (limit) 또는 전체 조회
        query = """
        MATCH (p:Property)
        OPTIONAL MATCH (p)-[r:NEAR_CULTURE]->(c:Culture)
        WHERE r.distance <= 500
        WITH p, count(c) as culture_count
        RETURN p.id as property_id, culture_count
        LIMIT $limit
        """
        with self.driver.session() as session:
            result = session.run(query, limit=limit)
            return pd.DataFrame([r.data() for r in result])

db = Neo4jConnector()
df = db.get_culture_counts(limit=10000)
db.close()

print(f"분석 대상 매물 수: {len(df)}")
display(df.head())

## 2. 분포 분석
매물 주변(500m) 문화시설 개수의 분포를 확인합니다.

In [ ]:
# 기술 통계
print(df['culture_count'].describe())

# 분포 시각화
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
sns.histplot(df['culture_count'], bins=20, kde=False)
plt.title('매물 주변(500m) 문화시설 개수 분포')
plt.xlabel('문화시설 개수')
plt.ylabel('매물 수')

plt.subplot(1, 2, 2)
sns.ecdfplot(data=df['culture_count'])
plt.title('누적 분포 (CDF)')
plt.xlabel('문화시설 개수')
plt.grid(True)

plt.tight_layout()
plt.show()

## 3. 점수화 로직 제안
- **개수 기반 점수**: 특정 개수 이상이면 만점(예: 3개 이상 100점).
- **구간별 점수**: 0개=0점, 1개=50점, 2개=80점, 3개 이상=100점 등.

### 분포 기반 제안
하위, 중위, 상위 분포를 보고 '적절한 만점 기준'을 정합니다.
예를 들어 상위 10%의 매물이 2개 이상의 시설을 가지고 있다면, 3개를 만점으로 설정하는 식입니다.

In [ ]:
# 분위수 확인
percentiles = [0.5, 0.75, 0.9, 0.95, 0.99]
print("=== 분위수별 개수 ===")
for p in percentiles:
    val = df['culture_count'].quantile(p)
    print(f"상위 {(1-p)*100:.0f}% (p{int(p*100)}): {val}개")

# 시뮬레이션: 만약 3개 이상이면 100점이라면?
max_criteria = 3
df['simulated_score'] = df['culture_count'].apply(lambda x: min(100, (x / max_criteria) * 100))

print(f"\n만점 기준 {max_criteria}개일 때 평균 점수: {df['simulated_score'].mean():.1f}점")
sns.histplot(df['simulated_score'], bins=10)
plt.title(f'만점 기준 {max_criteria}개일 때 점수 분포')
plt.show()